# Notebook 7: RC Circuits, Op-Amps, and a Bridge to Calculus

**Learning goal:** Model an RC transient, estimate slope and accumulated area numerically, and compare ideal integrator and differentiator behaviour without requiring physical hardware.

**Why it matters:** A reproducible calculation keeps the engineering assumptions, evidence, and explanation together.

**Prior knowledge:** Capacitors, RC time constants, voltage-time graphs, basic op-amp feedback, and NumPy arrays.

> A computed answer does not prove that the original circuit model, units, wiring, or measurements were correct. Use the checks and compare with physical evidence.


## Core and optional evidence

Every student can complete the model, numerical checks, graphs, and simulation comparison. A physical XL741 build or teacher demonstration is optional. If hardware is unavailable, label the result **modelled** or **simulated**, not measured.


## Working demonstration: RC charging

For a charging capacitor, $V_C(t)=V_S(1-e^{-t/RC})$. The time constant is $\tau=RC$. Predict the voltage at one time constant before running the cell.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

source_V = 5.0
resistance_ohm = 10_000.0
capacitance_F = 100e-6
tau_s = resistance_ohm * capacitance_F
time_s = np.linspace(0.0, 5.0 * tau_s, 501)
capacitor_V = source_V * (1.0 - np.exp(-time_s / tau_s))
index_tau = np.argmin(np.abs(time_s - tau_s))
print(f'Time constant: {tau_s:.3f} s')
print(f'Voltage at one time constant: {capacitor_V[index_tau]:.3f} V')
assert abs(capacitor_V[index_tau] / source_V - (1 - np.exp(-1))) < 1e-12

plt.plot(time_s, capacitor_V, label='Capacitor voltage')
plt.axvline(tau_s, color='tab:orange', linestyle='--', label='one time constant')
plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')
plt.title('RC charging model')
plt.grid(True)
plt.legend()
plt.show()


## From average slope to a derivative

A finite difference estimates $\Delta V/\Delta t$ between samples. `numpy.gradient` combines nearby differences to estimate the instantaneous slope. Compare it with the known derivative of the RC model.


In [ ]:
numerical_slope_V_per_s = np.gradient(capacitor_V, time_s)
model_slope_V_per_s = source_V / tau_s * np.exp(-time_s / tau_s)
largest_interior_error = np.max(np.abs(numerical_slope_V_per_s[1:-1] - model_slope_V_per_s[1:-1]))
print(f'Largest interior slope difference: {largest_interior_error:.5f} V/s')
assert largest_interior_error < 0.001

plt.plot(time_s, numerical_slope_V_per_s, label='Numerical slope')
plt.plot(time_s, model_slope_V_per_s, '--', label='Model derivative')
plt.xlabel('Time (s)')
plt.ylabel('Rate of voltage change (V/s)')
plt.title('Finite difference and derivative model')
plt.grid(True)
plt.legend()
plt.show()


## From accumulated rectangles to an integral

A running sum of $V\,\Delta t$ estimates area under a voltage-time graph. An ideal inverting op-amp integrator scales that accumulated area by $-1/(RC)$.


In [ ]:
sample_time_s = np.linspace(0.0, 4.0, 801)
dt_s = sample_time_s[1] - sample_time_s[0]
input_V = np.where((sample_time_s % 2.0) < 1.0, 1.0, -1.0)
accumulated_Vs = np.cumsum(input_V) * dt_s
integrator_R_ohm = 100_000.0
integrator_C_F = 10e-6
ideal_integrator_output_V = -accumulated_Vs / (integrator_R_ohm * integrator_C_F)
assert np.isfinite(ideal_integrator_output_V).all()

fig, axes = plt.subplots(2, 1, sharex=True)
axes[0].plot(sample_time_s, input_V)
axes[0].set_ylabel('Input (V)')
axes[0].grid(True)
axes[1].plot(sample_time_s, ideal_integrator_output_V)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Ideal output (V)')
axes[1].grid(True)
fig.suptitle('Square-wave input and ideal numerical integrator')
plt.show()


## Ideal differentiator and real limits

An ideal differentiator responds strongly at a sudden edge because the slope is very large. Real op-amps have finite bandwidth, slew rate, output range, noise, component tolerances, and supply limits. A numerical or ideal simulation is therefore a model to test, not proof of physical performance.


In [ ]:
input_slope_V_per_s = np.gradient(input_V, sample_time_s)
differentiator_R_ohm = 10_000.0
differentiator_C_F = 1e-6
ideal_differentiator_output_V = -(differentiator_R_ohm * differentiator_C_F) * input_slope_V_per_s
assert np.max(np.abs(ideal_differentiator_output_V)) > 0
plt.plot(sample_time_s, ideal_differentiator_output_V)
plt.xlabel('Time (s)')
plt.ylabel('Ideal output (V)')
plt.title('Numerical ideal differentiator response')
plt.grid(True)
plt.show()


## Optional XL741 voltage-follower pre-lab

The XL741 v2.0 assembly sheet suggests a simple unity-gain voltage-follower test with $+15$ V and $-15$ V supply rails and an input kept within $\pm13$ V. The model below is only an ideal system-level prediction. It does not simulate the kit's individual transistors, protection diodes, compensation capacitor, output limits, offset, bandwidth, or assembly faults. Verify the current manufacturer documentation and teacher-approved supply setup before any physical test.


In [ ]:
positive_supply_V = 15.0
negative_supply_V = -15.0
documented_test_input_limit_V = 13.0
follower_input_V = np.linspace(-documented_test_input_limit_V, documented_test_input_limit_V, 261)
ideal_follower_output_V = follower_input_V.copy()
assert ideal_follower_output_V.max() < positive_supply_V
assert ideal_follower_output_V.min() > negative_supply_V
assert np.allclose(ideal_follower_output_V, follower_input_V)

plt.plot(follower_input_V, ideal_follower_output_V, label='Ideal follower')
plt.plot(follower_input_V, follower_input_V, '--', label='Vout = Vin reference')
plt.xlabel('Input voltage (V)')
plt.ylabel('Output voltage (V)')
plt.title('XL741 follower pre-lab: ideal prediction only')
plt.grid(True)
plt.legend()
plt.show()


## Purposeful problem

Choose new teacher-approved resistor and capacitor values. Predict the new time constant, verify the 63.2% charging point, and determine how the initial slope changes. Add assertions before interpreting the graph.


In [ ]:
# Student code: change R and C, calculate tau, generate the model, and check V(tau).
student_R_ohm = 22_000.0
student_C_F = 47e-6
# Add calculations, labelled graphs, and at least two independent checks.


## Evidence table

Complete every available column. Leave **Measured** marked *not available* when no physical kit or instrument is used; do not copy simulated values into that column.

| Quantity | Hand estimate | Python model | Circuit simulation | Measured, optional |
|---|---:|---:|---:|---:|
| RC time constant (s) |  |  |  |  |
| Voltage at one time constant (V) |  |  |  |  |
| Initial slope (V/s) |  |  |  |  |
| Integrator output change (V) |  |  |  |  |


## Independent challenge

Compare the same input four ways when possible: hand estimate, Python numerical model, circuit simulation, and optional XL741 measurement. Explain one difference caused by sampling and one difference a real op-amp could introduce.

### Hints

1. Find slope with two nearby points before using derivative notation.
2. Find accumulated area with rectangles before using integral notation.
3. Reduce the time step and see whether the numerical result converges.
4. Never claim the model validates soldering, wiring, ratings, or hardware safety.


## Common errors

- Check volts versus millivolts, amperes versus milliamperes, and ohms versus kilohms.
- Run cells in order after restarting the kernel.
- Read the last line of an error first and confirm every filename and variable name.
- Keep full precision for intermediate values and round only the reported result.


## Reflection

What do you know, what do you need to find out, and how will you know when you are right? Explain what your check established and one limitation it could not remove.
